# Simple Store - Pure Python Website (PyWebIO) in Jupyter

In [16]:
%pip install pywebio

Note: you may need to restart the kernel to use updated packages.


## 1. Store data and functions

In [17]:
from pywebio.input import input, actions, input_group, NUMBER
from pywebio.output import put_text, put_table, put_markdown, clear
from pywebio import start_server

# Product data stored as parallel lists (basically Python's version of arrays)
product_names = ["Notebook", "Pen", "Bag", "Water Bottle", "Calculator"]
product_prices = [50, 10, 500, 150, 300]
product_stock = [20, 100, 15, 30, 10]

# Cart data - starts empty
cart_items = []
cart_prices = []


def show_products():
    rows = [["No.", "Product", "Price", "Stock"]]
    for i in range(len(product_names)):
        rows.append([i + 1, product_names[i], product_prices[i], product_stock[i]])
    put_table(rows)


def buy_product():
    options = []
    for i in range(len(product_names)):
        label = product_names[i] + " (Price: " + str(product_prices[i]) + ", Stock: " + str(product_stock[i]) + ")"
        options.append(label)
    options.append("Cancel")

    choice = actions("Choose a product to buy", options)
    if choice == "Cancel":
        return

    index = options.index(choice)
    quantity = input("Enter quantity", type=NUMBER)

    if quantity <= 0:
        put_text("Quantity must be more than 0.")
        return

    if quantity > product_stock[index]:
        put_text("Not enough stock available!")
        return

    product_stock[index] = product_stock[index] - quantity
    total_price = product_prices[index] * quantity
    cart_items.append(product_names[index] + " x" + str(quantity))
    cart_prices.append(total_price)

    put_text(product_names[index] + " added to cart. Price: " + str(total_price))


def sell_product():
    data = input_group("Add New Product", [
        input("Product Name", name="name"),
        input("Price", name="price", type=NUMBER),
        input("Stock Quantity", name="stock", type=NUMBER),
    ])

    product_names.append(data["name"])
    product_prices.append(data["price"])
    product_stock.append(data["stock"])

    put_text(data["name"] + " has been added to the store!")


def show_cart():
    if len(cart_items) == 0:
        put_text("Your cart is empty.")
        return

    rows = [["Item", "Price"]]
    total = 0
    for i in range(len(cart_items)):
        rows.append([cart_items[i], cart_prices[i]])
        total = total + cart_prices[i]

    put_table(rows)
    put_text("Total Amount: " + str(total))


def main():
    while True:
        clear()
        put_markdown("# Simple Store")
        show_products()

        choice = actions("What would you like to do?", [
            "Buy a Product",
            "Sell / Add a Product",
            "View Cart",
            "Exit",
        ])

        if choice == "Buy a Product":
            buy_product()
        elif choice == "Sell / Add a Product":
            sell_product()
        elif choice == "View Cart":
            show_cart()
        elif choice == "Exit":
            clear()
            put_text("Thank you for visiting the store!")
            break

## 2. Start the server (runs in the background so this notebook stays usable)

In [18]:
import threading
import asyncio

def run_app():
    # PyWebIO's server needs its own event loop when started on a
    # background thread instead of the main thread
    asyncio.set_event_loop(asyncio.new_event_loop())
    start_server(main, port=8080, debug=True)

threading.Thread(target=run_app, daemon=True).start()
print("Server started! Open http://127.0.0.1:8080 in your browser.")

Server started! Open http://127.0.0.1:8080 in your browser.


Exception in thread Thread-20 (run_app):
Traceback (most recent call last):
  File "d:\Anaconda\Lib\threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "d:\Anaconda\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "d:\Anaconda\Lib\threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\SHIV\AppData\Local\Temp\ipykernel_12916\3799061150.py", line 8, in run_app
    start_server(main, port=8080, debug=True)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Lib\site-packages\pywebio\platform\tornado.py", line 291, in start_server
    _, port = _setup_server(webio_handler=handler, port=port, host=host, static_dir=static_dir,
              ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                    

**To stop the server:** use **Kernel > Restart Kernel** when you're done — there's no clean Ctrl+C for a background thread inside a notebook.